In [1]:
# 1. Imports y rutas

import os
import cv2
import xml.etree.ElementTree as ET
from tqdm import tqdm

BASE_DIR = "Datos cartas"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

OUT_TRAIN = "dataset/train"
OUT_TEST  = "dataset/test"

os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)


In [2]:
# 2. Función para procesar una carpeta (leer XML y recortar)

def process_folder(src_dir, out_dir):
    for file in tqdm(os.listdir(src_dir)):
        if not file.endswith(".xml"):
            continue

        xml_path = os.path.join(src_dir, file)
        tree = ET.parse(xml_path)
        root = tree.getroot()

        filename = root.find("filename").text
        img_path = os.path.join(src_dir, filename)

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)

        obj = root.find("object")
        label = obj.find("name").text.replace(" ", "_")

        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        crop = img[ymin:ymax, xmin:xmax]

        class_dir = os.path.join(out_dir, label)
        os.makedirs(class_dir, exist_ok=True)

        cv2.imwrite(os.path.join(class_dir, filename), crop)


In [3]:
# 3. Crear dataset de entrenamiento y test

process_folder(TRAIN_DIR, OUT_TRAIN)
process_folder(TEST_DIR, OUT_TEST)


100%|███████████████████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 234.83it/s]


In [4]:
# 4. Cargar datos con Keras

import tensorflow as tf
# To use tensorflow we can't use python 3.13, we create a new enviroment with an older version
# conda create -n tf_env python=3.10
# conda activate tf_env
# conda install -c conda-forge tensorflow

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    OUT_TRAIN,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_gen = test_datagen.flow_from_directory(
    OUT_TEST,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_gen.num_classes
print("Clases:", train_gen.class_indices)


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 5. Modelo CNN

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(base_model.input, output)

model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# 6. Entrenamiento

history = model.fit(
    train_gen,
    epochs=15
)


In [ ]:
# 7. Evaluación en test

test_loss, test_acc = model.evaluate(test_gen)
print("Accuracy en test:", test_acc)


In [ ]:
# Predicción de una carta correcta

import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "Datos cartas/test/201.jpg"

img = image.load_img(img_path, target_size=IMG_SIZE)
x = image.img_to_array(img) / 255.0
x = np.expand_dims(x, axis=0)

pred = model.predict(x)
pred_idx = np.argmax(pred)

labels = {v: k for k, v in train_gen.class_indices.items()}
print("Carta predicha:", labels[pred_idx])
